<a href="https://colab.research.google.com/github/x1001000/Colab-Notebooks/blob/main/gemini_2_5_pro_%E8%AE%80_NVDA_%E8%B2%A1%E5%A0%B1_PDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PDF 上傳 Gemini Files API

In [ ]:
pdf_url = 'https://nvidianews.nvidia.com/_gallery/download_pdf/6837703d3d63320fddb3a9ee/'

import requests
pdf_content = requests.get(pdf_url).content

import io
pdf_file_object = io.BytesIO(pdf_content)

from google import genai
from google.genai.types import Tool, GenerateContentConfig
from google.colab import userdata
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
uploaded_file = client.files.upload(file=pdf_file_object, config={'mime_type': 'application/pdf'})

# 提示詞一行 + keys

In [8]:
prompt = '''Output values in JSON format.

keys:
company_name
quarter
period_end_date
currency
units
revenue
cost_of_revenue
gross_profit
operating_expenses
research_and_development_expenses
selling_general_and_administrative_expenses
operating_income
interest_income
interest_expense
income_before_tax
income_tax_expense
net_income
eps
eps_diluted
total_current_assets
cash_and_cash_equivalents
short_term_investments
cash_and_short_term_investments
net_receivables
inventory
total_non_current_assets
property_plant_equipment_net
goodwill_and_intangible_assets
total_assets
total_current_liabilities
account_payables
short_term_debt
total_non_current_liabilities
long_term_debt
total_liabilities
total_debt
net_debt
preferred_stock
common_stock
retained_earnings
total_equity
net_cash_provided_by_operating_activities
net_cash_used_for_investing_activities
net_cash_used_provided_by_financing_activities
depreciation_and_amortization
free_cash_flow
net_change_in_cash
effect_of_forex_change_on_cash
'''

# 同時十次 API call

In [27]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

def call_gemini_api():
    response = client.models.generate_content(
        model='gemini-2.5-pro',
        contents=[prompt, uploaded_file],
        config=GenerateContentConfig(
            response_mime_type='application/json',
            response_schema = str
        )
    )
    return response

async def call_gemini_api_async():
    loop = asyncio.get_event_loop()
    with ThreadPoolExecutor() as executor:
        response = await loop.run_in_executor(executor, call_gemini_api)
    return response

async def main_concurrent():
    tasks = [call_gemini_api_async() for _ in range(10)]
    responses = await asyncio.gather(*tasks)
    return responses

responses = await main_concurrent()

# parse API 回傳的 JSON、有快取的不採用

In [29]:
data = []
for response in responses:
    if response.usage_metadata.model_dump().get('cached_content_token_count'):
        print('有一次快取')
    else:
        try:
            import json
            data.append(json.loads(response.parsed))
        except:
            print('有一次 json.loads(response.parsed) 失敗')

import pandas as pd
df = pd.DataFrame(data)

有一次快取


# 找眾數 (含 NaN)

In [30]:
def get_mode_with_nan(series):
    # Count all values, including NaN
    counts = series.value_counts(dropna=False)
    if counts.empty:
        return None # Or some other indicator for empty series
    # Get the value(s) with the maximum count
    max_count = counts.max()
    modes = counts[counts == max_count].index.tolist()
    # Return the first mode if multiple, or handle as needed
    return modes[0] if modes else None

# Calculate the mode for each column including NaN
modes_with_nan = df.apply(get_mode_with_nan)

# You can now see the modes including NaN
# display(modes_with_nan.to_frame().T)

# If you still want to append this mode to the DataFrame and display transposed with highlight
# Calculate the mode including NaN and take the first row
modes_row_with_nan = modes_with_nan

# Append the modes as a new row to the DataFrame using pd.concat
# Need to reset index to concatenate the row
df_with_mode_with_nan = pd.concat([df.reset_index(drop=True), modes_row_with_nan.to_frame().T], ignore_index=True)

# Display the transpose of the DataFrame with the modes row and rename the last index
df_with_mode_T_with_nan = df_with_mode_with_nan.T
# Rename the last column of the transposed DataFrame
df_with_mode_T_with_nan = df_with_mode_T_with_nan.rename(columns={df_with_mode_T_with_nan.columns[-1]: "眾數 (含 NaN)"})
# display(df_with_mode_T_with_nan)

# Re-apply the highlighting with the new mode calculation
def highlight_diff_with_nan_mode(row):
    modes = row['眾數 (含 NaN)']
    is_diff = (row != modes)
    # Handle NaN comparison specifically - NaN != NaN is True, but we don't want to highlight it as a difference from mode if mode is also NaN
    is_diff = is_diff & ~(pd.isna(row) & pd.isna(modes))
    # Don't highlight the mode column itself
    is_diff['眾數 (含 NaN)'] = False
    return ['background-color: yellow' if v else '' for v in is_diff]

df_with_mode_T_with_nan.style.apply(highlight_diff_with_nan_mode, axis=1)

,0,1,2,3,4,5,6,7,8,眾數 (含 NaN)
company_name,NVIDIA Corporation,NVIDIA,NVIDIA,NVIDIA,NVIDIA,NVIDIA,NVIDIA,NVIDIA,NVIDIA,NVIDIA
quarter,Q1,Q1,Q1,Q1,Q1 FY26,Q1,Q1,Q1,Q1,Q1
period_end_date,2025-04-27,2025-04-27,2025-04-27,2025-04-27,2025-04-27,2025-04-27,2025-04-27,2025-04-27,2025-04-27,2025-04-27
currency,USD,USD,USD,USD,USD,USD,USD,USD,USD,USD
units,millions,millions,millions,Millions,millions,Millions,millions,Millions,millions,millions
revenue,44062,44062,44062,44062,44062,44062,44062,44062,44062,44062
cost_of_revenue,17394,17394,17394,17394,17394,17394,17394,17394,17394,17394
gross_profit,26668,26668,26668,26668,26668,26668,26668,26668,26668,26668
operating_expenses,5030,5030,5030,5030,5030,5030,5030,5030,5030,5030
research_and_development_expenses,3989,3989,3989,3989,3989,3989,3989,3989,3989,3989
